# Custom CNN Multi-label Classification with LIME


> **Reproducibility note:** Textual training outputs are retained from the original course run, while selected figures are preserved under assets/. Credential uploads, extraction logs, and local paths were removed. Rerunning can produce different values.


In [ ]:
from pathlib import Path
import os

SEED = 42

def find_repository_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "config").is_dir():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Start Jupyter from the repository root."
    )

project_root_override = os.getenv("JTS_PROJECT_ROOT")
REPO_ROOT = (
    Path(project_root_override).expanduser().resolve()
    if project_root_override
    else find_repository_root()
)

def configured_path(variable: str, default: Path) -> Path:
    value = os.getenv(variable)
    path = Path(value).expanduser() if value else default
    if not path.is_absolute():
        path = REPO_ROOT / path
    return path.resolve()

DATASET_ROOT = configured_path("JTS_DATASET_ROOT", REPO_ROOT / "data" / "Final_Dataset")
MODELS_DIR = REPO_ROOT / "models"
ASSETS_DIR = REPO_ROOT / "assets"
MANUAL_TEST_DIR = ASSETS_DIR / "manual-tests"

def require_directory(path: Path, purpose: str) -> Path:
    if not path.is_dir():
        raise FileNotFoundError(f"{purpose} directory not found: {path}")
    return path

def require_file(path: Path, purpose: str) -> Path:
    if not path.is_file():
        raise FileNotFoundError(f"{purpose} file not found: {path}")
    return path

MODEL_PATH = configured_path("JTS_CNN_MODEL", MODELS_DIR / "custom_cnn.keras")


In [ ]:
import random
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from lime import lime_image
from skimage.segmentation import mark_boundaries
from tensorflow.keras import layers

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [ ]:
required_dataset_directories = [
    DATASET_ROOT / category / split
    for category in ("images", "labels")
    for split in ("train", "val", "test")
]
for directory in required_dataset_directories:
    require_directory(directory, "Dataset split")
print(f"Using dataset: {DATASET_ROOT}")


The dataset is external. Follow `data/README.md` to download it from Kaggle.

Do not place `kaggle.json` in this repository. Use Kaggle's user-level configuration or the `KAGGLE_USERNAME` and `KAGGLE_KEY` environment variables.


In [ ]:
manual_images = {
    1: require_file(MANUAL_TEST_DIR / "hump.png", "Manual test image"),
    3: require_file(MANUAL_TEST_DIR / "pedestrian_cross.png", "Manual test image"),
    4: require_file(MANUAL_TEST_DIR / "no_stop.png", "Manual test image"),
    6: require_file(MANUAL_TEST_DIR / "pass_either.png", "Manual test image"),
}


# Preparing Data

In [ ]:
CLASS_NAMES = [
    "No_Entry",
    "Hump",
    "Stop",
    "Pedestrian_Cross",
    "No_Stop",
    "Give_Way",
    "Pass_Either",
]

def prepare_data(image_directory, label_directory):
    image_directory = require_directory(Path(image_directory), "Image")
    label_directory = require_directory(Path(label_directory), "Label")

    image_by_stem = {
        path.stem: path
        for path in sorted(image_directory.iterdir())
        if path.is_file()
    }
    label_by_stem = {
        path.stem: path
        for path in sorted(label_directory.glob("*.txt"))
    }
    common_stems = sorted(image_by_stem.keys() & label_by_stem.keys())
    if not common_stems:
        raise ValueError(
            f"No matching images and labels found in {image_directory} and {label_directory}."
        )

    rows = []
    for stem in common_stems:
        labels = [0] * len(CLASS_NAMES)
        with label_by_stem[stem].open("r", encoding="utf-8") as handle:
            for line in handle:
                if not line.strip():
                    continue
                class_id = int(line.split()[0])
                if class_id not in range(len(CLASS_NAMES)):
                    raise ValueError(f"Unexpected class {class_id} in {label_by_stem[stem]}")
                labels[class_id] = 1
        rows.append({"Path": str(image_by_stem[stem]), **dict(zip(CLASS_NAMES, labels))})

    return pd.DataFrame(rows)


In [ ]:
training_df = prepare_data(
    DATASET_ROOT / "images" / "train",
    DATASET_ROOT / "labels" / "train",
)
training_df.head(2)


                                             Path  No_Entry  Hump  Stop  \
0  data/Final_Dataset/images/train/Bg1268.png         0     1     0   
1   data/Final_Dataset/images/train/Ag368.png         0     0     0   

   Pedestrian_Cross  No_Stop  Give_Way  Pass_Either  
0                 0        0         0            0  
1                 0        0         0            1  

In [ ]:
validation_df = prepare_data(
    DATASET_ROOT / "images" / "val",
    DATASET_ROOT / "labels" / "val",
)


# Build CNN model

In [ ]:
def extract_val(df):
    img = df['Path'].values 
    labels = df.drop('Path',axis=1).values 
    df2 = tf.data.Dataset.from_tensor_slices((img, labels))
    return df2

In [ ]:
train_dataset = extract_val(training_df)
valid_dataset = extract_val(validation_df)

In [ ]:
def preprocess(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [224, 224])
    return image, tf.cast(label, tf.float32)


In [ ]:
batch_size = 64
train_ds = (
    train_dataset.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(128, seed=SEED, reshuffle_each_iteration=True)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)
valid_ds = (
    valid_dataset.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)


In [ ]:
model = tf.keras.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D((2, 2), strides=2),
    layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D((2, 2), strides=2),
    layers.Dropout(0.5),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(7, activation="sigmoid"),
])


In [ ]:
model.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    metrics=["accuracy"],
)

hist = model.fit(train_ds, validation_data=valid_ds, epochs=10)
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
model.save(MODEL_PATH)


Epoch 1/10
109/109 [==============================] - 488s 4s/step - loss: 23.9659 - accuracy: 0.1696 - val_loss: 0.4489 - val_accuracy: 0.1824
Epoch 2/10
109/109 [==============================] - 487s 4s/step - loss: 0.4020 - accuracy: 0.3106 - val_loss: 0.4413 - val_accuracy: 0.2397
Epoch 3/10
109/109 [==============================] - 483s 4s/step - loss: 0.3406 - accuracy: 0.4320 - val_loss: 0.4407 - val_accuracy: 0.2754
Epoch 4/10
109/109 [==============================] - 485s 4s/step - loss: 0.2896 - accuracy: 0.5304 - val_loss: 0.4374 - val_accuracy: 0.3888
Epoch 5/10
109/109 [==============================] - 486s 4s/step - loss: 0.2505 - accuracy: 0.6094 - val_loss: 0.4502 - val_accuracy: 0.5009
Epoch 6/10
109/109 [==============================] - 479s 4s/step - loss: 0.2200 - accuracy: 0.6805 - val_loss: 0.4217 - val_accuracy: 0.5619
Epoch 7/10
109/109 [==============================] - 484s 4s/step - loss: 0.2035 - accuracy: 0.7123 - val_loss: 0.3770 - val_accuracy: 0.592

In [ ]:
# Accuracy and loss values figures
fig = plt.figure(figsize=(14,4))
fig.add_subplot(121)
plt.plot(hist.history['accuracy'])
plt.plot(hist.history['val_accuracy'])
plt.legend(['Training accuracy', 'Validation accuracy'])
plt.title('Training and validation accuracy')
plt.xlabel('Epoch Number')
plt.ylabel('Accuracy')

# Loss Figure
fig.add_subplot(122)
plt.plot(hist.history['loss'])
plt.plot(hist.history['val_loss'])
plt.legend(['Training loss', 'Validation loss'])
plt.title('Training and validation loss')
plt.xlabel('Epoch Number')
plt.ylabel('Loss')

plt.show()

<Figure size 1008x288 with 2 Axes>

# XAI .. LIME

Run the training cell above, or set the model environment variable to a compatible local checkpoint. Checkpoints are not stored in Git.


In [ ]:
loaded_model = tf.keras.models.load_model(
    require_file(MODEL_PATH, "Saved model")
)


In [ ]:
for expected_class, image_path in manual_images.items():
    require_file(image_path, f"Manual test image for class {expected_class}")


In [ ]:
def load_image(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

In [ ]:
image1 = load_image(manual_images[1])
image3 = load_image(manual_images[3])
image4 = load_image(manual_images[4])
image6 = load_image(manual_images[6])


In [ ]:
def preprocess_img(image, image_size):
    return cv2.resize(image, (image_size, image_size)).astype(np.float32)


In [ ]:
def explain_image_lime(image, model):
    resized = preprocess_img(image, 224)
    explainer = lime_image.LimeImageExplainer(random_state=SEED)
    explanation = explainer.explain_instance(
        resized.astype("double"),
        lambda batch: model.predict(batch, verbose=0),
        top_labels=5,
        hide_color=0,
        num_samples=1000,
        random_seed=SEED,
    )
    _, mask = explanation.get_image_and_mask(
        explanation.top_labels[0],
        positive_only=False,
        num_features=10,
        hide_rest=False,
    )
    return mark_boundaries(resized.astype(np.uint8), mask)


In [ ]:
f, axarr = plt.subplots(2,2,figsize=(12, 12))
axarr[0,0].imshow(explain_image_lime(image1, loaded_model))
axarr[0,1].imshow(explain_image_lime(image3, loaded_model))
axarr[1,0].imshow(explain_image_lime(image4, loaded_model))
axarr[1,1].imshow(explain_image_lime(image6, loaded_model))

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

<Figure size 864x864 with 4 Axes>

In [ ]:
def predict_class(path, model):
    image = preprocess_img(load_image(path), 224)
    outputs = model.predict(np.expand_dims(image, axis=0), verbose=0)
    return int(np.argmax(outputs, axis=1)[0])


In [ ]:
for expected_class, image_path in manual_images.items():
    predicted_class = predict_class(image_path, loaded_model)
    print(f"Actual class: {expected_class}. Predicted class: {predicted_class}")


Acutal Class: 1 .. Prediction Class: 5
Acutal Class: 3 .. Prediction Class: 5
Acutal Class: 4 .. Prediction Class: 4
Acutal Class: 6 .. Prediction Class: 6
